In [2]:
# for each word, use spacy to get its stem or lemma, save in a list
import spacy
import re
import json
import ast
from pathlib import Path
import pandas as pd
from tqdm import tqdm
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "textcat"])
from collections import Counter
# from transformers import GPT2TokenizerFast, AutoTokenizer

In [3]:
# lexical items involved in Maya's evaluation set

transitive_verbs = ["accept", "acknowledge", "admit", "aggravate", "answer", "arrest", "ask", "avoid", "bash", "beat", "bend", "bite", "bless", "bother", "break", "brush", "build", "bump", "burn", "call", "cancel", "capture", "carry", "catch", "change", "charge", "chase", "chastise", "check", "chill", "clean", "close", "clutch", "collect", "comfort", "confuse", "consume", "contradict", "convert", "copy", "correct", "cover", "crack", "cross", 'cut', "dampen", "dash", "daze", "dazzle", "deceive", "define", "delay", "deny", "derail", "describe", "destroy", "devastate", "dig", "discover", "discuss", "dismiss", "distinguish", 'disturb', 'drag', 'draw', 'dress', 'drink', 'drive', 'drop', 'drown', 'dry', 'dunk', 'eat', 'edify', 'eject', 'embarrass', 'embrace', 'empower', 'enable', 'enclose', 'encourage', 'enjoy', 'enlighten', 'enlist', 'entertain', 'escort', 'examine', 'excite', 'excuse', 'execute', 'fascinate', 'feed', 'feel', 'fight', 'file', 'fill', 'find', 'finish', 'fire', 'fix', 'flick', 'flip', 'follow', 'force', 'forget', 'forgive', 'freeze', 'frighten', 'fry', 'furnish', 'gather', 'get', 'grab', 'grasp', 'grease', 'grip', 'handle', 'hang', 'have', 'head', 'help', 'hide', 'highlight', 'hit', 'hoist', 'hold', 'honor', 'hug', 'hurry', 'hurt', 'imitate', 'impress', 'include', 'indulge', 'inform', 'insert', 'inspect', 'inspire', 'insure', 'interest', 'interrupt', 'interview', 'intimidate', 'involve', 'irritate', 'join', 'jolt', 'judge', 'keep', 'key', 'kick', 'kill', 'kiss', 'knock', 'lag', 'lay', 'lead', 'lean', 'leave', 'let', 'lick', 'lift', 'light', 'lighten', 'limit', 'link', 'list', 'load', 'lock', 'lose', 'love', 'lower', 'maintain', 'make', 'mark', 'marry', 'massage', 'melt', 'mix', 'mock', 'move', 'munch', 'name', 'notice', 'number', 'nurse', 'offend', 'open', 'order', 'own', 'pack', 'page', 'paralyze', 'park', 'pass', 'pay', 'persuade', 'petrify', 'pick', 'pierce', 'pin', 'place', 'play', 'please', 'poison', 'poke', 'possess', 'post', 'pour', 'prepare', 'press', 'print', 'promise', 'protect', 'pull', 'punch', 'punish', 'purchase', 'push', 'puzzle', 'question', 'quit', 'raid', 'raise', 'read', 'reassure', 'recognize', 'refill', 'relax', 'remind', 'remove', 'repel', 'replace', 'research', 'retard', 'retire', 'reveal', 'ride', 'ring', 'rip', 'rob', 'rub', 'run', 'satisfy', 'save', 'scan', 'scare', 'scold', 'scoop', 'scrub', 'seat', 'see', 'select', 'sell', 'send', 'set', 'sew', 'shake', 'shame', 'shift', 'shoot', 'shove', 'shut', 'sink', 'slam', 'slap', 'slice', 'slow', 'smell', 'smoke', 'snap', 'soak', 'soften', 'solve', 'sound', 'specify', 'speed', 'spell', 'spend', 'spill', 'spit', 'split', 'spoon', 'spread', 'squash', 'stab', 'stain', 'stake', 'start', 'startle', 'stay', 'steer', 'stir', 'stop', 'store', 'strike', 'study', 'stuff', 'suck', 'surprise', 'survey', 'swallow', 'switch', 'tape', 'taste', 'teach', 'tease', 'tell', 'tend', 'terrify', 'test', 'thank', 'threaten', 'throw', 'tickle', 'tie', 'tighten', 'tip', 'tire', 'toast', 'toss', 'touch', 'toe', 'transform', 'try', 'turn', 'tweak', 'twist', 'underestimate', 'understand', 'unload', 'unlock', 'untie', 'upgrade', 'use', 'vacate', 'videotape', 'vilify', 'violate', 'wake', 'want', 'warm', 'warn', 'wash', 'watch', 'wear', 'widen', 'win', 'wipe', 'wrack', 'wrap', 'wreck']

intransitive_verbs = ['quivered', 'faded', 'rested', 'proceeded', 'reflected', 'led', 'belonged', 'progressed', 'differed', 'yielded', 'glowed', 'interfered', 'began', 'whistled', 'voted', 'arose', 'fled', 'buzzed', 'volunteered', 'delighted', 'landed', 'yawned', 'acted', 'evolved', 'erupted', 'emerged', 'withdrew', 'squeaked', 'attended', 'partied', 'winked', 'rejoiced', 'leaped', 'shouted', 'flourished', 'roared', 'objected', 'intervened', 'insisted', 'cheered', 'lingered', 'stood', 'interrupted', 'trembled', 'apologized', 'recovered', 'escaped', 'fell', 'prevailed', 'faltered', 'sneezed', 'froze', 'listened', 'consented', 'jumped', 'appeared', 'exploded', 'relented', 'protested', 'complained', 'vanished', 'arrived', 'pounced', 'screamed', 'obeyed', 'yelped', 'gasped', 'tired', 'moved', 'persisted', 'paused', 'relaxed', 'hesitated', 'survived', 'giggled', 'collapsed', 'came', 'blinked', 'shivered', 'rose', 'grinned', 'cried', 'blushed', 'disappeared', 'quit', 'frowned', 'lied', 'groaned', 'sighed', 'stopped', 'succeeded', 'existed', 'laughed', 'smiled', 'nodded', 'agreed']#subcases for transitive verbs depending on whether they take animate, inanimate, neutral

animate = ['told', 'fed', 'paid', 'protected', 'trusted', 'challenged', 'adored', 'questioned', 'convinced', 'promised', 'encouraged', 'invited', 'mocked', 'guided', 'advised', 'complimented', 'rescued', 'instructed', 'reminded', 'informed', 'hired', 'greeted', 'appointed', 'entertained', 'rewarded', 'punished', 'blessed', 'summoned', 'assured', 'forgave', 'thanked', 'accompanied', 'escorted', 'persuaded', 'employed', 'notified', 'honored', 'scolded', 'interviewed', 'cautioned', 'befriended', 'reassured', 'congratulated', 'hugged', 'chastised', 'alerted']

inanimate = ['made', 'got', 'felt', 'wanted', 'cut', 'tried', 'built', 'wrote', 'enjoyed', 'bought', 'mentioned', 'fixed', 'experienced', 'explained', 'posted', 'designed', 'realized', 'threw', 'reported', 'opened', 'spread', 'ordered', 'shared', 'understood', 'denied', 'defined', 'pulled', 'sold', 'wore', 'caught', 'changed', 'filled', 'tied', 'handled', 'presented', 'prepared', 'rewrote', 'noticed', 'proposed', 'dropped', 'stated', 'tasted', 'published', 'purchased', 'locked', 'imagined', 'split', 'updated', 'dismissed', 'connected', 'admitted', 'printed', 'studied', 'rejected', 'passed', 'recorded', 'justified', 'remembered', 'loaded', 'boiled', 'sewed', 'grabbed', 'delivered', 'chopped', 'measured', 'packed', 'polished', 'cleaned', 'acknowledged', 'solved', 'edited', 'specified', 'debated']

inanimate_with_objects = [['made', 'a cake'], ['got', 'the gift'], ['felt', 'the cloth'], ['wanted', 'the gift'], ['cut', 'the cake'], ['tried', 'the dish'], ['built', 'the tower'], ['wrote', 'the book'], ['enjoyed', 'the show'], ['bought', 'the gift'], ['mentioned', 'the event'], ['fixed', 'the bike'], ['experienced', 'the show'], ['explained', 'the problem'], ['posted', 'the picture'], ['designed', 'the experiment'], ['realized', 'the dream'], ['threw', 'the ball'], ['reported', 'the story'], ['opened', 'the door'], ['spread', 'the news'], ['ordered', 'the package'], ['shared', 'the candy'], ['understood', 'the problem'], ['defined', 'the word'], ['denied', 'permission'], ['pulled', 'the rope'], ['sold', 'the book'], ['wore', 'the clothes'], ['caught', 'the ball'], ['changed', 'the color'], ['tied', 'the rope'], ['filled', 'the bucket'], ['handled', 'the problem'], ['presented', 'the talk'], ['prepared', 'dinner'], ['rewrote', 'the book'], ['noticed', 'the flower'], ['proposed', 'the idea'], ['dropped', 'the ball'], ['stated', 'the rule'], ['tasted', 'the food'], ['published', 'the book'], ['purchased', 'the book'], ['locked', 'the door'], ['imagined', 'the sunrise'], ['split', 'the apple'], ['updated', 'the post'], ['dismissed', 'the comment'], ['connected', 'the dots'], ['admitted', 'the mistake'], ['printed', 'the poster'], ['studied', 'the book'], ['rejected', 'the suggestion'], ['passed', 'the test'], ['recorded', 'the song'], ['justified', 'the action'], ['remembered', 'the song'], ['loaded', 'the car'], ['boiled', 'the water'], ['sewed', 'the shirt'], ['grabbed', 'the keys'], ['delivered', 'the package'], ['chopped', 'the vegetables'], ['measured', 'the height'], ['packed', 'the suitcase'], ['polished', 'the shoes'], ['cleaned', 'the closet'], ['acknowledged', 'the gift'], ['solved', 'the problem'], ['edited', 'the book'], ['specified', 'the rules'], ['debated', 'the issue']]

animate_present = ['call', 'devastate', 'forgive', 'reassure', 'startle', 'confuse', 'mock', 'rob', 'interview', 'disturb', 'judge', 'punish', 'imitate', 'empower', 'impress', 'fire', 'protect', 'chastise', 'offend', 'thank', 'marry', 'enlighten', 'comfort', 'intimidate', 'arreste', 'bless', 'scare', 'inform', 'contradict', 'enlist', 'destroy', 'kiss', 'hug', 'vilify', 'repel', 'join', 'persuade', 'aggravate', 'deceive', 'warn', 'remind', 'help', 'escort', 'question', 'tell', 'irritate', 'embarrass', 'bother', 'scold', 'surprise', 'interrupt', 'frighten', 'threaten', 'honor']

inanimate_present = ['lighten', 'print', 'spill', 'change', 'spend', 'cancel', 'understand', 'dry', 'order', 'pour', 'prepare', 'wear', 'want', 'sell', 'spread', 'try', 'pack', 'specify', 'build', 'copy', 'fix', 'taste', 'tighten', 'smell', 'get', 'split', 'notice', 'drink', 'admit', 'solve', 'widen', 'make', 'store', 'purchase', 'feel', 'enclose', 'pass', 'refill', 'post', 'grasp', 'pull', 'sew', 'maintain', 'study', 'enjoy']

verbs_with_that = ["accepted", "announced", "checked", "considered", "decided", "discovered", "forgot", "guessed", "imagined", "knew", "noticed", "proved", "remembered", "said", "saw", "understood"]

verbs_with_what = ["announced", "discovered", "forgot", "guessed", "knew", "remembered", "saw"]

infinitival_verbs = ["decided", "forgot", "remembered"]

nouns = ["you", "I", "we", "they", "he", "she", "it", "the doctor", "the scientist", "the person", "the singer", "the teacher", "the student", "the parent", "the child", "the writer", "the artist", "the friend", "the sibling", "John", "Mary", "Alex", "the neighbor", "Louis", "Catherine", "the astronaut"]

objects = ["you", "me", "us", "them", "him", "her", "it", "the doctor", "the scientist", "the person", "the singer", "the teacher", "the student", "the parent", "the child", "the writer", "the artist", "the friend", "the sibling", "John", "Mary", "Alex", "the neighbor", "Louis", "Catherine", "the astronaut"]

def remove_brackets(sentence: str) -> str:
    # Remove [ ... ] including the brackets
    cleaned = re.sub(r"\[.*?\]", "", sentence)
    # Collapse extra whitespace
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    # Fix stray spaces before punctuation
    cleaned = re.sub(r"\s+([?.!,])", r"\1", cleaned)
    return cleaned

Verb_Categories = [transitive_verbs, 
                   intransitive_verbs, 
                   animate, 
                   inanimate, 
                   animate_present, 
                   inanimate_present, 
                   verbs_with_that, 
                   verbs_with_what, 
                   infinitival_verbs]

Noun_Categories = [[item.split()[-1] for item in nouns], 
                   [item.split()[-1] for item in objects],
                   [item[1].split()[-1] for item in inanimate_with_objects]]

Construction_Categories = {'MQ': ['SMQ', 'OMQ', 'CC_SMQ', 'CC_OMQ'],
                           'EQ': ['SEQ', 'OEQ'],
                           'RC': ['SRC', 'ORC', 'SRC_reduced', 'ORC_reduced']}
Construction_Flat = ['SMQ', 'OMQ', 'CC_SMQ', 'CC_OMQ', 'SEQ', 'OEQ', 'SRC', 'ORC', 'SRC_reduced', 'ORC_reduced',
                     'MQ', 'EQ', 'RC',
                     'None_Construction', 'All_Sentences']

# Outline

Ultimately, we want: for each [dataset] x [child, parent] (they are mutually exclusive), get a dictionary that:
- each key is a verb, each value is a dictionary;
- within each verb's dictionary: each key is a condition, and each value is the number of occurrence in that condition;
- conditions: 
    - Level 0: 'SMQ', 'OMQ', 'CC_SMQ', 'CC_OMQ', 'SEQ', 'OEQ', 'SRC', 'ORC', 'SRC_reduced', 'ORC_reduced';
    - Level 1: 'MQ', 'EQ', 'RC'
    - Level 2: 'None_Construction', 'All_Sentences'

Also, we need a Verb <-> Category mapping:
- for each unique verb: get the set of categories it belongs to;
- for each category: given a set of rare verbs, see how many of them are from this category (get a percentage); 

In [4]:
# get the set of unique words from the lists above
unique_verbs = set()
for word_list in Verb_Categories:
    for word in word_list:
        unique_verbs.add(word)
# convert the set back to a list
unique_verbs = list(unique_verbs)
len(unique_verbs)

576

In [5]:
def load_dataset(dataset, child_filtered, construction):
    '''
    Load the dataset and filter based on speaker and construction type.
    `construction` is either a string (level 0 or 2) or a list of strings (level 1).
    '''
    file_path = f'all_labeled_data/data_Nov15/LABELED_{dataset}.csv'
    df = pd.read_csv(file_path)
    df['labels'] = df['labels'].apply(ast.literal_eval)
    
    if child_filtered:
        df = df[df['speaker'] == 'PAR']
    else:
        df = df[df['speaker'] == 'CHI']
        
    if construction == 'All_Sentences': # "all" means get all sentences, no filtering
        utterances = df['sentence_clean'].tolist()
    elif construction == 'None_Construction': # "none" means sentences without any target construction
        # utterances = df[df['labels'] == []]['sentence_clean'].tolist()
        utterances = df[df['labels'].apply(lambda x: len(x) == 0)]['sentence_clean'].tolist()

    else:
        if isinstance(construction, str):
            construction = [construction]
        utterances = []
        for _, row in df.iterrows():
            labels = row['labels']
            if any(label in construction for label in labels):
                utterances.append(row['sentence_clean'])
    return utterances
    
def count_all_word_frequency(utterances):
    counts = Counter()
    # text_clean = [remove_brackets(utt) for utt in utterances]
    
    all_text = " ".join(utterances)
    all_text = re.sub(r"[^\w\s]", " ", all_text)  # remove punctuation
    all_words = all_text.split()  # split into words
    counts.update(Counter(all_words))  # get the raw frequency of each unique word in the utterances

    return counts

def extract_lexical_frequency(counts, unique_words, json_path, condition_label):
    json_path = Path(json_path)
    json_path.parent.mkdir(parents=True, exist_ok=True)

    # Load existing JSON if it exists and is non-empty; otherwise start fresh
    if not json_path.exists() or json_path.stat().st_size == 0:
        data = {}
    else:
        with json_path.open("r", encoding="utf-8") as f:
            data = json.load(f)

    # Update entries
    for word in unique_words:
        frequency = counts.get(word, 0)
        if word not in data or not isinstance(data[word], dict):
            data[word] = {}
        data[word][condition_label] = frequency

    # Save updated JSON
    with json_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)

In [6]:
for dataset in ['train_10M']:#, 'dev', 'test', 'train_100M'
    for child_filtered in [True, False]:# , False
        file_child = 'Child' if not child_filtered else 'noChild'
        print(f"#################### On Dataset {dataset}, Filtered {child_filtered} #################### ")
        
        for construction in Construction_Flat[-2:]:
            if construction in ['MQ', 'EQ', 'RC']: target_construction = Construction_Categories[construction]
            else: target_construction = construction
            
            json_path = f'dataset_lexical_frequency/EvalTokens_{dataset}_{file_child}.json'
            utterances = load_dataset(dataset, child_filtered, target_construction)
            all_counts = count_all_word_frequency(utterances)
            extract_lexical_frequency(all_counts, unique_verbs, json_path, construction)
            print(f"finished construction {construction}")
            
            if construction == 'All_Sentences':
                with open(f'dataset_lexical_frequency/AllTokens_{dataset}_{file_child}.json', 'w') as f:
                    json.dump(all_counts, f)
            

#################### On Dataset train_10M, Filtered True #################### 
finished construction None_Construction
finished construction All_Sentences
#################### On Dataset train_10M, Filtered False #################### 
finished construction None_Construction
finished construction All_Sentences


### Check threshold

# Archived

In [ ]:
def count_word_frequency(utterances, method):
    counts = Counter()
    text_clean = [remove_brackets(utt) for utt in utterances]
    
    if method == 'tokenizer_based':
        tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
        for sentence in tqdm(text_clean):
            token_ids = tokenizer(sentence).input_ids
            counts.update(token_ids)
    elif method == 'spacy_lemmatized':
        # for sentence in tqdm(text_clean):
        #     doc = nlp(sentence)
        #     lemmas = [token.lemma_.lower() for token in doc]
        #     counts.update(lemmas)
        for doc in tqdm(nlp.pipe(text_clean, batch_size=1000)):
            for tok in doc:
                if tok.is_space or tok.is_punct or not tok.text.isalpha():
                    continue
                counts[tok.lemma_.lower()] += 1
    elif method == 'raw_word_freq':
        all_text = " ".join(text_clean)
        all_text = re.sub(r"[^\w\s]", " ", all_text)  # remove punctuation
        # all_text = all_text.lower()  # convert to lowercase
        all_words = all_text.split()  # split into words
        counts.update(Counter(all_words))  # get the raw frequency of each unique word in the utterances

    return counts

In [ ]:
for method in ['spacy_lemmatized', 'tokenizer_based', 'raw_word_freq']:
    for dataset in ['dev', 'test', 'train_10M', 'train_100M']:
        for child_filtered in [False, True]:
            utterances = load_dataset(dataset, child_filtered)
            counts = count_word_frequency(utterances, method)
            out_path = Path(f"dataset_lexical_frequency/{method}/{dataset}_{child_filtered}.json")
            out_path.parent.mkdir(parents=True, exist_ok=True)
            with out_path.open("w", encoding="utf-8") as f:
                json.dump(dict(counts), f, ensure_ascii=False, indent=2)

### Aggregate results and check with threshold

In [4]:
# assume we only want to look at files with Adult speech
def aggregate_counting_results(method, child_filtered, dataset=None):
    files = [str(p) for p in Path(f"dataset_lexical_frequency/{method}/").rglob("*.json") if str(child_filtered) in p.name and (dataset is None or dataset in p.name)]
    print(f"Currently looking at the following files: {files}")

    merged = Counter()

    for fp in files:
        p = Path(fp)
        if not p.exists():
            print(f"Skip missing: {p}")
            continue
        with p.open(encoding="utf-8") as f:
            data = json.load(f)          # expects {"eat": 50, "protest": 11, ...}
            merged.update({k: int(v) for k, v in data.items()})  # ensure ints

    merged_sorted = dict(merged.most_common())
    return merged_sorted

def prepare_eval_words(eval_words, method):
    eval_units = []
    
    if method == 'spacy_lemmatized':
        for word in eval_words:
            doc = nlp(word)
            for token in doc:
                eval_units.append(token.lemma_)
        eval_units = list(set(eval_units))
    
    elif method == 'tokenizer_based':
        tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
        for word in eval_words:
            token_ids = tokenizer(word).input_ids
            eval_units.extend(token_ids)
        eval_units = list(set(eval_units)) # this is currently problematic if eval_words has multi-token words
        
    elif method == 'raw_word_freq':
        eval_units = eval_words
        
    return eval_units

def get_eval_word_frequencies(freq_dic, eval_units, threshold=10):
    return {word: freq_dic.get(word, 0) for word in eval_units if freq_dic.get(word, 0) < threshold}

In [ ]:
method='raw_word_freq'
freq_dic = aggregate_counting_results(method=method, child_filtered=True)

eval_units = prepare_eval_words(unique_words, method=method)
low_freq_words = get_eval_word_frequencies(freq_dic, eval_units, threshold=10)
print(low_freq_words)

Currently looking at the following files: ['dataset_lexical_frequency/raw_word_freq/dev_True.json', 'dataset_lexical_frequency/raw_word_freq/test_True.json', 'dataset_lexical_frequency/raw_word_freq/train_100M_True.json', 'dataset_lexical_frequency/raw_word_freq/train_10M_True.json']
{'empower': 0, 'deceive': 3, 'lag': 2, 'execute': 0, 'erupted': 0, 'scolded': 5, 'the door': 0, 'the closet': 0, 'chastise': 0, 'reveal': 7, 'emerged': 4, 'the car': 0, 'specified': 0, 'obeyed': 1, 'evolved': 1, 'the problem': 0, 'soften': 3, 'edify': 0, 'the tower': 0, 'dismissed': 1, 'admitted': 2, 'attended': 1, 'the water': 0, 'intimidate': 1, 'instructed': 4, 'pierce': 9, 'insure': 0, 'cautioned': 0, 'questioned': 2, 'denied': 7, 'hoist': 4, 'specify': 3, 'the cloth': 0, 'define': 6, 'the sibling': 0, 'maintain': 8, 'updated': 2, 'the food': 0, 'tweak': 2, 'indulge': 1, 'withdrew': 0, 'the dream': 0, 'the neighbor': 0, 'escort': 1, 'retire': 9, 'partied': 5, 'lighten': 4, 'deny': 8, 'convert': 1, 'obj